# 🔑 Notebook 1: Naive Hashing Problems

## What You'll Learn

In this notebook, we'll explore three approaches to distributing data across servers:

| Approach | Quality | Key Redistribution When Resizing |
|----------|---------|----------------------------------|
| Modulo hashing | ❌ BAD | ~75% of keys move |
| Simple hash ring | ✅ BETTER | ~20-30% of keys move |
| Hash ring + virtual nodes | 🏆 BEST | ~20% move, **balanced** load |

> **Real-world motivation:** Imagine you're building a cache for a ticketing system.
> You have 3 cache servers. When you add a 4th server to handle more traffic,
> you don't want to invalidate almost every cached ticket!

In [ ]:
import hashlib
import bisect
from collections import defaultdict, Counter

## Helper: A Simple Hash Function

We'll use MD5 to convert any string into a number. This gives us a deterministic,
well-distributed number for any input.

In [ ]:
def hash_key(key: str) -> int:
    """Convert a string key into a large integer using MD5."""
    return int(hashlib.md5(key.encode()).hexdigest(), 16)

# Quick demo
for key in ["user:1", "user:2", "user:3"]:
    h = hash_key(key)
    print(f"  hash_key('{key}') = {h}")

---

## ❌ BAD: Modulo Hashing

The simplest way to distribute keys across `N` servers:

```
server_index = hash(key) % num_servers
```

This is like dealing cards — key 1 goes to server 1, key 2 to server 2, etc.
It works perfectly... **until the number of servers changes.**

```
        ┌─────────┐
 key ▶  │ hash()  │ ▶ hash_value % 3 ▶ Server 0, 1, or 2
        └─────────┘
```

In [ ]:
def modulo_hash(key: str, num_servers: int) -> int:
    """Assign a key to a server using modulo hashing."""
    return hash_key(key) % num_servers

# Distribute 10,000 keys across 3 servers
NUM_KEYS = 10_000
num_servers = 3

assignments = {}
for i in range(NUM_KEYS):
    key = f"event:{i}"
    assignments[key] = modulo_hash(key, num_servers)

# Count keys per server
distribution = Counter(assignments.values())
print("📊 Key distribution across 3 servers:")
for server in sorted(distribution):
    count = distribution[server]
    bar = "█" * (count // 100)
    print(f"  Server {server}: {count:,} keys  {bar}")
print(f"\n✅ Looks great! Keys are roughly evenly distributed.")

### 💥 Problem 1: Adding a Server

Business is booming! Let's add a 4th server. We just change `% 3` to `% 4`, right?

Let's see what happens to our 10,000 keys...

In [ ]:
# Now hash the SAME keys with 4 servers instead of 3
new_num_servers = 4
keys_moved = 0

for i in range(NUM_KEYS):
    key = f"event:{i}"
    old_server = modulo_hash(key, num_servers)      # % 3
    new_server = modulo_hash(key, new_num_servers)   # % 4
    if old_server != new_server:
        keys_moved += 1

pct = keys_moved / NUM_KEYS * 100
print(f"💥 Adding 1 server (3 → 4):")
print(f"   Keys that moved: {keys_moved:,} / {NUM_KEYS:,} ({pct:.1f}%)")
print()
print(f"   ❌ Almost {pct:.0f}% of keys need to move to a different server!")
print(f"   That means {pct:.0f}% of cache misses, {pct:.0f}% of data to migrate.")
print(f"   Your database just got slammed with traffic. 🔥")

### 💥 Problem 2: Removing a Server

Now imagine a server crashes. We go from 3 servers to 2.

In [ ]:
# Hash the SAME keys with 2 servers instead of 3
fewer_servers = 2
keys_moved = 0

for i in range(NUM_KEYS):
    key = f"event:{i}"
    old_server = modulo_hash(key, num_servers)    # % 3
    new_server = modulo_hash(key, fewer_servers)   # % 2
    if old_server != new_server:
        keys_moved += 1

pct = keys_moved / NUM_KEYS * 100
print(f"💥 Removing 1 server (3 → 2):")
print(f"   Keys that moved: {keys_moved:,} / {NUM_KEYS:,} ({pct:.1f}%)")
print()
print(f"   ❌ A server goes down AND ~{pct:.0f}% of remaining keys are wrong!")
print(f"   Double trouble: you lost a server AND your cache is mostly invalid.")

### 📋 Why Modulo Hashing Fails

| Scenario | Keys That Move |
|----------|---------------|
| Add 1 server (3→4) | ~75% |
| Remove 1 server (3→2) | ~67% |
| Add 1 server (10→11) | ~91% |

**The math:** When changing from `N` to `M` servers, roughly `1 - min(N,M)/max(N,M)` × 100% of keys move.
More servers = worse redistribution!

> We need an approach where adding or removing a server only affects a **small fraction** of keys.

---

## ✅ BETTER: Simple Consistent Hash Ring

Instead of `hash(key) % N`, we arrange servers on a **circular ring** (0 to 2³² - 1).
Each key is assigned to the **next server clockwise** on the ring.

```
         0
         │
    S3 ──┼── S1      Servers placed at fixed positions on the ring.
         │            Keys "walk clockwise" to find their server.
         │
         S2
```

**Key insight:** When you add or remove a server, only the keys between
that server and its counter-clockwise neighbor need to move.
Everything else stays put! 🎯

In [ ]:
class SimpleHashRing:
    """A consistent hash ring WITHOUT virtual nodes."""

    def __init__(self):
        self.ring = {}          # hash_value -> server_name
        self.sorted_keys = []   # sorted list of hash positions on the ring

    def add_node(self, node: str):
        """Place a server at one position on the ring."""
        h = hash_key(node)
        self.ring[h] = node
        bisect.insort(self.sorted_keys, h)

    def remove_node(self, node: str):
        """Remove a server from the ring."""
        h = hash_key(node)
        del self.ring[h]
        self.sorted_keys.remove(h)

    def get_node(self, key: str) -> str:
        """Find which server a key belongs to (walk clockwise)."""
        if not self.ring:
            return None
        h = hash_key(key)
        # Find the first server position >= the key's hash
        idx = bisect.bisect_right(self.sorted_keys, h)
        # Wrap around to the beginning if we passed the end (it's a ring!)
        if idx == len(self.sorted_keys):
            idx = 0
        return self.ring[self.sorted_keys[idx]]

print("✅ SimpleHashRing class defined!")

### Testing the Simple Hash Ring

In [ ]:
# Create a ring with 3 servers
ring = SimpleHashRing()
for server in ["Server-A", "Server-B", "Server-C"]:
    ring.add_node(server)

# Distribute the same 10,000 keys
ring_assignments = {}
for i in range(NUM_KEYS):
    key = f"event:{i}"
    ring_assignments[key] = ring.get_node(key)

distribution = Counter(ring_assignments.values())
print("📊 Key distribution on hash ring (3 servers):")
for server in sorted(distribution):
    count = distribution[server]
    bar = "█" * (count // 100)
    print(f"  {server}: {count:,} keys  {bar}")
print()
max_keys = max(distribution.values())
min_keys = min(distribution.values())
print(f"  ⚠️  Imbalance: largest server has {max_keys/min_keys:.1f}x more keys than smallest")
print(f"     (We'll fix this with virtual nodes later!)")

### Adding a Server to the Ring

In [ ]:
# Add a 4th server
ring.add_node("Server-D")

# Re-assign all keys
keys_moved = 0
for i in range(NUM_KEYS):
    key = f"event:{i}"
    new_server = ring.get_node(key)
    if ring_assignments[key] != new_server:
        keys_moved += 1

pct = keys_moved / NUM_KEYS * 100
print(f"Adding Server-D to the ring:")
print(f"   Keys that moved: {keys_moved:,} / {NUM_KEYS:,} ({pct:.1f}%)")
print()
print(f"   ✅ Only ~{pct:.0f}% of keys moved! (vs ~75% with modulo hashing)")
print(f"   Most keys stayed on the same server. 🎉")

### Removing a Server from the Ring

In [ ]:
# Reset: rebuild with 3 servers
ring2 = SimpleHashRing()
for server in ["Server-A", "Server-B", "Server-C"]:
    ring2.add_node(server)

original = {}
for i in range(NUM_KEYS):
    key = f"event:{i}"
    original[key] = ring2.get_node(key)

# Remove Server-B (simulating a crash)
ring2.remove_node("Server-B")

keys_moved = 0
for i in range(NUM_KEYS):
    key = f"event:{i}"
    new_server = ring2.get_node(key)
    if original[key] != new_server:
        keys_moved += 1

pct = keys_moved / NUM_KEYS * 100
print(f"Removing Server-B from the ring:")
print(f"   Keys that moved: {keys_moved:,} / {NUM_KEYS:,} ({pct:.1f}%)")
print()
print(f"   ✅ Only the keys that were ON Server-B moved!")
print(f"   Servers A and C kept all their existing keys.")

### ⚠️ The Remaining Problem: Unbalanced Load

The simple hash ring minimizes key movement — great! But load isn't evenly distributed.
With only 3 positions on a ring of 2³² slots, some servers end up with
way more keys than others.

We need a way to spread each server's "territory" more evenly around the ring.

---

## 🏆 BEST: Virtual Nodes

The fix is simple but powerful: instead of placing each server at **one** position
on the ring, place it at **many** positions using different labels.

```
Without virtual nodes:          With virtual nodes (3 per server):
         0                               0
         │                          A1 ──┼── B2
    C ───┼── A                   C3      │      A2
         │                      B1 ──────┼────── C1
         B                          A3 ──┴── B3
(uneven territories)            (much more balanced!)
```

For example, Server-A gets positions for `Server-A#0`, `Server-A#1`, `Server-A#2`, etc.
More virtual nodes = more even distribution. In practice, **100–200** virtual nodes per server is common.

In [ ]:
class VirtualNodeHashRing:
    """A consistent hash ring WITH virtual nodes for balanced distribution."""

    def __init__(self, num_virtual_nodes: int = 150):
        self.num_virtual_nodes = num_virtual_nodes
        self.ring = {}           # hash_value -> physical server name
        self.sorted_keys = []
        self.nodes = set()       # track physical nodes

    def add_node(self, node: str):
        """Place a server at multiple positions using virtual node labels."""
        self.nodes.add(node)
        for i in range(self.num_virtual_nodes):
            virtual_key = f"{node}#vn{i}"
            h = hash_key(virtual_key)
            self.ring[h] = node
            bisect.insort(self.sorted_keys, h)

    def remove_node(self, node: str):
        """Remove all virtual nodes for a server."""
        self.nodes.discard(node)
        for i in range(self.num_virtual_nodes):
            virtual_key = f"{node}#vn{i}"
            h = hash_key(virtual_key)
            if h in self.ring:
                del self.ring[h]
                self.sorted_keys.remove(h)

    def get_node(self, key: str) -> str:
        """Find which server a key belongs to (walk clockwise)."""
        if not self.ring:
            return None
        h = hash_key(key)
        idx = bisect.bisect_right(self.sorted_keys, h)
        if idx == len(self.sorted_keys):
            idx = 0
        return self.ring[self.sorted_keys[idx]]

print("✅ VirtualNodeHashRing class defined!")

In [ ]:
# Create a ring with 3 servers, 150 virtual nodes each
vring = VirtualNodeHashRing(num_virtual_nodes=150)
for server in ["Server-A", "Server-B", "Server-C"]:
    vring.add_node(server)

# Distribute 10,000 keys
vring_assignments = {}
for i in range(NUM_KEYS):
    key = f"event:{i}"
    vring_assignments[key] = vring.get_node(key)

distribution = Counter(vring_assignments.values())
print("📊 Key distribution with virtual nodes (150 per server):")
for server in sorted(distribution):
    count = distribution[server]
    bar = "█" * (count // 100)
    print(f"  {server}: {count:,} keys  {bar}")
print()
max_keys = max(distribution.values())
min_keys = min(distribution.values())
ideal = NUM_KEYS // len(distribution)
print(f"  Ideal per server: {ideal:,}")
print(f"  Max/Min ratio: {max_keys/min_keys:.2f}x")
print(f"  ✅ Much more balanced than the simple ring!")

In [ ]:
# Test: Add a 4th server
vring.add_node("Server-D")

keys_moved = 0
for i in range(NUM_KEYS):
    key = f"event:{i}"
    new_server = vring.get_node(key)
    if vring_assignments[key] != new_server:
        keys_moved += 1

pct = keys_moved / NUM_KEYS * 100
print(f"Adding Server-D (with virtual nodes):")
print(f"   Keys that moved: {keys_moved:,} / {NUM_KEYS:,} ({pct:.1f}%)")
print(f"   ✅ Only ~{pct:.0f}% moved — and the load from moved keys is spread evenly!")
print()

# Check new distribution
new_assignments = {}
for i in range(NUM_KEYS):
    key = f"event:{i}"
    new_assignments[key] = vring.get_node(key)

distribution = Counter(new_assignments.values())
print("📊 New distribution across 4 servers:")
for server in sorted(distribution):
    count = distribution[server]
    bar = "█" * (count // 100)
    print(f"  {server}: {count:,} keys  {bar}")

In [ ]:
# Test: Remove Server-B (simulating a crash)
vring2 = VirtualNodeHashRing(num_virtual_nodes=150)
for server in ["Server-A", "Server-B", "Server-C"]:
    vring2.add_node(server)

original_v = {}
for i in range(NUM_KEYS):
    key = f"event:{i}"
    original_v[key] = vring2.get_node(key)

vring2.remove_node("Server-B")

keys_moved = 0
moved_to = Counter()
for i in range(NUM_KEYS):
    key = f"event:{i}"
    new_server = vring2.get_node(key)
    if original_v[key] != new_server:
        keys_moved += 1
        moved_to[new_server] += 1

pct = keys_moved / NUM_KEYS * 100
print(f"Removing Server-B (with virtual nodes):")
print(f"   Keys that moved: {keys_moved:,} / {NUM_KEYS:,} ({pct:.1f}%)")
print()
print(f"   Where Server-B's keys went:")
for server, count in sorted(moved_to.items()):
    print(f"     → {server}: {count:,} keys")
print()
print(f"   ✅ Server-B's load is spread across remaining servers, not dumped on one!")

---

## 📊 Grand Comparison: BAD → BETTER → BEST

In [ ]:
print("=" * 65)
print(f"{'Approach':<30} {'Add Server':>15} {'Remove Server':>15}")
print("=" * 65)

# BAD: Modulo hashing
add_moved = sum(1 for i in range(NUM_KEYS)
                if modulo_hash(f"event:{i}", 3) != modulo_hash(f"event:{i}", 4))
rem_moved = sum(1 for i in range(NUM_KEYS)
                if modulo_hash(f"event:{i}", 3) != modulo_hash(f"event:{i}", 2))
label1 = "❌ Modulo hashing"
print(f"{label1:<30} {add_moved/NUM_KEYS*100:>14.1f}% {rem_moved/NUM_KEYS*100:>14.1f}%")

# BETTER: Simple ring
ring_a = SimpleHashRing()
for s in ["Server-A", "Server-B", "Server-C"]:
    ring_a.add_node(s)
orig = {f"event:{i}": ring_a.get_node(f"event:{i}") for i in range(NUM_KEYS)}
ring_a.add_node("Server-D")
add_moved = sum(1 for i in range(NUM_KEYS) if orig[f"event:{i}"] != ring_a.get_node(f"event:{i}"))

ring_b = SimpleHashRing()
for s in ["Server-A", "Server-B", "Server-C"]:
    ring_b.add_node(s)
orig2 = {f"event:{i}": ring_b.get_node(f"event:{i}") for i in range(NUM_KEYS)}
ring_b.remove_node("Server-B")
rem_moved = sum(1 for i in range(NUM_KEYS) if orig2[f"event:{i}"] != ring_b.get_node(f"event:{i}"))
label2 = "✅ Simple hash ring"
print(f"{label2:<30} {add_moved/NUM_KEYS*100:>14.1f}% {rem_moved/NUM_KEYS*100:>14.1f}%")

# BEST: Virtual nodes
vr_a = VirtualNodeHashRing(150)
for s in ["Server-A", "Server-B", "Server-C"]:
    vr_a.add_node(s)
orig3 = {f"event:{i}": vr_a.get_node(f"event:{i}") for i in range(NUM_KEYS)}
vr_a.add_node("Server-D")
add_moved = sum(1 for i in range(NUM_KEYS) if orig3[f"event:{i}"] != vr_a.get_node(f"event:{i}"))

vr_b = VirtualNodeHashRing(150)
for s in ["Server-A", "Server-B", "Server-C"]:
    vr_b.add_node(s)
orig4 = {f"event:{i}": vr_b.get_node(f"event:{i}") for i in range(NUM_KEYS)}
vr_b.remove_node("Server-B")
rem_moved = sum(1 for i in range(NUM_KEYS) if orig4[f"event:{i}"] != vr_b.get_node(f"event:{i}"))
label3 = "🏆 Virtual nodes (150)"
print(f"{label3:<30} {add_moved/NUM_KEYS*100:>14.1f}% {rem_moved/NUM_KEYS*100:>14.1f}%")

print("=" * 65)
print()
print("💡 Key takeaway: Virtual nodes give us BOTH minimal redistribution")
print("   AND balanced load distribution. That's why real systems use them!")

## 🎯 Key Takeaways

1. **Modulo hashing** (`hash % N`) is simple but causes massive key redistribution when `N` changes
2. **Consistent hash ring** arranges servers on a circle — only neighboring keys move during changes
3. **Virtual nodes** solve the load imbalance problem by giving each server multiple positions on the ring
4. Real-world systems like DynamoDB and Cassandra use consistent hashing with virtual nodes

## ⏭️ Next Up

In **Notebook 2**, we'll build a production-quality hash ring class from scratch
and **visualize** how keys are distributed around the ring using matplotlib.